[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/19_playbook_elite/19_playbook_elite.ipynb)

# Módulo 19 · Playbook de élite: decisiones, trampas y revisión

**Nivel:** 🔴 Experto · **Duración:** ≈ 4 h (lección ≈ 2.5 h + proyecto ≈ 1.5 h) · **GPU:** no necesaria (CPU, `FAST_DEV_RUN`) · **Cómputo estimado:** 0 unidades de Colab.

**Prerrequisitos:** todo el curso, en especial 04 (vocabularios), 12 (embedders), 13 (híbrido), 14 (ANN), 15 (evaluación) y 18 (producción).

> 🧭 **Ruta núcleo vs ampliación.** Núcleo (≈ 1.5 h): §1 árboles de decisión · §2 trampas de evaluación · §5 galería de fallos · §6 checklist. Ampliación (≈ 1 h): §3 gap offline/online · §4 costos a escala · §7 cultura.

**Hilo conductor — Políglota.** Políglota Labs ha crecido: hay tres equipos, dos clientes enterprise y una entrevista de *staff engineer* la semana que viene. Las lecciones anteriores **enumeraban** buenas prácticas; aquí las convertimos en **código que decide, mide y revisa**: árboles de decisión ejecutables, trampas de evaluación reproducidas con números, una calculadora de costos a escala, una galería de fallos reales reproducidos y un *linter* de PRs. Este módulo produce además `extras/GUIA_ENTREVISTAS.md` y `extras/LECTURA_PAPERS.md`.

> **Estado del arte a octubre de 2026.** Las reglas de los árboles son **heurísticas razonadas** con su fuente o con una medición en este notebook; los umbrales concretos dependen de tu hardware y de tus datos: re-mídelos.

## 🎯 Objetivos de aprendizaje
1. **Codificar** árboles de decisión para tokenizer, embedder, índice e híbrido, y **justificar** sus umbrales con mediciones.
2. **Reproducir y cuantificar** cuatro trampas de evaluación: tuning en test, *winner's curse*, dependencia del golden set y sesgo de *pooling*.
3. **Explicar** el gap offline/online y **recalcular** la comparación de sistemas con la mezcla real de tráfico.
4. **Estimar** el costo total de un sistema de retrieval de 1 M a 1 B de documentos (embedding, memoria, réplicas) con supuestos explícitos.
5. **Diagnosticar** fallos reales documentados y reproducir versiones mínimas de los más comunes.
6. **Revisar** PRs de tokenización/embeddings con una checklist ejecutable.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1 · (M14)</b> ¿Hasta qué tamaño basta la fuerza bruta?</summary>

Hasta donde su latencia cumpla el SLA con batch = 1: depende de $N \cdot d$ y del hardware. Lo **mediremos** en §1 en vez de citar un número.
</details>
<details><summary><b>2 · (M15)</b> ¿Por qué no se elige un hiperparámetro mirando el test?</summary>

Porque el máximo de muchas estimaciones ruidosas está sesgado hacia arriba: el número reportado es optimista. Lo cuantificamos en §2 con validación cruzada.
</details>
<details><summary><b>3 · (M18)</b> ¿Qué ventaja tiene interleaving sobre A/B y qué no mide?</summary>

Mucha más potencia por impresión (comparación pareada dentro de la misma impresión); no mide impacto en métricas de negocio de sesión, que siguen pidiendo A/B.
</details>
<details><summary><b>4 · (M05)</b> ¿Por qué aparecen tokens «glitch»?</summary>

Porque el tokenizer se entrenó con datos distintos de los del modelo: hay tokens que el modelo casi nunca vio y cuyos embeddings quedaron sin entrenar (Land & Bartolo 2024).
</details>

## 🗺️ Mapa de madurez (a oct-2026) de las decisiones de este playbook
🟢 sólido · 🟡 emergente · 🔴 experimental.

| Decisión / técnica | Madurez | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| BM25 como baseline obligatorio | 🟢 | BEIR lo mostró fuerte fuera de dominio | Thakur et al. 2021 |
| Fusión híbrida (RRF / lineal) | 🟢 | Elasticsearch/OpenSearch, Qdrant, Weaviate, Vespa | Cormack et al. 2009; Bruch et al. 2022 |
| Reranker cross-encoder en 2ª etapa | 🟢 | práctica estándar en búsqueda y RAG | Nogueira & Cho 2019 |
| Bi-encoder multilingüe abierto + fine-tuning de dominio | 🟢 | e5, BGE-M3, Qwen3-Embedding, EmbeddingGemma | módulos 11–12 |
| HNSW en RAM / IVF-PQ / DiskANN en SSD | 🟢 | FAISS, Lucene, pgvector, Qdrant, DiskANN | módulo 14 |
| Cuantización binaria + rescoring | 🟡 | sentence-transformers, varias vector DBs | Shakir et al. 2024 |
| Late interaction (ColBERT/PLAID/MUVERA) | 🟡 | ColBERTv2, PLAID; MUVERA (Google Research) | módulo 13 |
| LLM-as-judge para qrels | 🟡 | útil con validación humana; sesgos conocidos | Thomas et al. 2023; Soboroff 2024 |
| Tokenizer-free (BLT, H-Net) para retrieval | 🔴 | investigación | módulos 06–07 |
| Límites de dimensión (LIMIT) como criterio de diseño | 🟡 | — | Weller et al. 2025 |

## 0. Preparación
Las mismas piezas canónicas del curso (corpus, métricas, BM25, estadística pareada, embedders v1/v2 con fallback) y el mini golden set de **paráfrasis** ES↔EN↔código del módulo 13.

In [ ]:
!pip install -q faiss-cpu qdrant-client fastapi httpx sentence-transformers scikit-learn matplotlib pandas

In [ ]:
import os, sys, time, math, random, json, hashlib, unicodedata, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True       # True: tamaños diminutos (CPU en minutos). False en Colab con GPU para modelos reales y más datos.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
try:
    import torch
    torch.manual_seed(SEED); torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, device = None, "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """IC percentil de la media remuestreando QUERIES (la unidad de muestreo es la query, no el par query-doc)."""
    v = np.asarray(values, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def paired_bootstrap(a, b, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42) -> dict:
    """Test pareado A vs B sobre las MISMAS queries. IC percentil de la diferencia media y p-valor bilateral
    por 'shift' (se centra la diferencia en 0 bajo H0 y se cuenta cuántas veces el remuestreo supera lo observado)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    null = (d - d.mean())[idx].mean(axis=1)
    p = (np.sum(np.abs(null) >= abs(d.mean())) + 1) / (n_boot + 1)
    return {"diff": float(d.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "p_value": float(p)}

def paired_permutation(a, b, n_perm: int = 10000, seed: int = 42) -> float:
    """Test de aleatorización pareado (sign-flip): bajo H0 el signo de cada diferencia por query es intercambiable."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(d)))
    stat = np.abs((signs * d).mean(axis=1))
    return float((np.sum(stat >= abs(d.mean()) - 1e-12) + 1) / (n_perm + 1))

def holm(pvalues: dict, alpha: float = 0.05) -> dict:
    """Corrección de Holm-Bonferroni para comparaciones múltiples. Devuelve {nombre: (p_ajustado, rechaza_H0)}."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for i, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[name] = (running, running < alpha)
    return out

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]; doc_texts = [d["text"] for d in docs]
doc_lang = np.array([d["lang"] for d in docs])
qids = [q["qid"] for q in queries]; q_texts = [q["text"] for q in queries]; q_lang = np.array([q["lang"] for q in queries])
print(f"fuente={corpus['source']} · {len(docs)} docs · {len(queries)} queries · idiomas={sorted(set(doc_lang))}")

In [ ]:
def run_desde_scores(S, k: int = 10, ids=None) -> dict:
    """Matriz de scores (n_queries x n_docs) -> run {qid: [doc_ids top-k]} con el desempate canónico."""
    ids = doc_ids if ids is None else ids
    return {qid: rank_from_scores(S[i], ids, k) for i, qid in enumerate(qids)}

def ndcg_por_query(run: dict, k: int = 10) -> np.ndarray:
    """nDCG@k de cada query (orden de `qids`); unidad de remuestreo para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), qrels[q], k) for q in qids])

def l2n(X):
    X = np.asarray(X, dtype=np.float32)
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)

BM25_IDX = BM25().fit([tokenize_simple(t) for t in doc_texts], doc_ids)
S_BM25 = np.array([BM25_IDX.scores(tokenize_simple(q)) for q in q_texts])
print("BM25 (baseline del módulo 00):", {m: round(v, 3) for m, v in evaluate(run_desde_scores(S_BM25), qrels).items() if m in ("ndcg", "recall", "mrr")})

In [ ]:
# Dos versiones del embedder de Políglota: v1 (la que está en producción) y v2 (la candidata).
# Con hub: v1 = multilingual-e5-small (384 d), v2 = multilingual-e5-base (768 d).
# Sin hub: dos fallbacks TF-IDF+SVD DISTINTOS (otro analizador y otra dimensión) -> espacios incompatibles, como en la vida real.
USAR_MODELOS_REALES = True
V1_REAL, V2_REAL = "multilingual-e5-small", "multilingual-e5-base"

def carga_versiones(textos_ajuste) -> dict:
    vers = {}
    if USAR_MODELOS_REALES:
        for nombre, clave in (("v1", V1_REAL), ("v2", V2_REAL)):
            e = cargar_embedder(clave)          # None si no hay hub
            if e is None:
                break
            vers[nombre] = e
    if len(vers) < 2:
        print("⚠️  Sin modelos reales: v1/v2 = fallbacks TF-IDF+SVD (NO son modelos neuronales; las cifras no representan a e5).")
        vers = {"v1": Embedder("v1 · char 3-4 + SVD 48d (fallback)", {}, fallback=HashSVDEmbedder(dim=48, ngram=(3, 4)).fit(textos_ajuste)),
                "v2": Embedder("v2 · char 2-5 + SVD 128d (fallback)", {}, fallback=HashSVDEmbedder(dim=128, ngram=(2, 5)).fit(textos_ajuste))}
    return vers

VERS = carga_versiones(doc_texts)
EMB = {}
for v, e in VERS.items():
    t0 = time.perf_counter()
    D, Qv = e.encode_docs(doc_texts), e.encode_queries(q_texts)
    EMB[v] = {"D": D, "Q": Qv, "seg": time.perf_counter() - t0}
    print(f"{v}: {e.clave} · dim={D.shape[1]} · nDCG@10={evaluate(run_desde_scores(Qv @ D.T), qrels)['ndcg']:.3f} · {EMB[v]['seg']:.1f}s")

In [ ]:
# Mini-corpus sintético "Políglota": 29 conceptos técnicos x {EN, ES, código}. Es el FALLBACK del cargador de datos;
# tiene el mismo esquema que el corpus real. Las queries están parafraseadas a propósito para que no compartan
# casi ninguna palabra con el documento (así BM25 no es trivial y los embeddings multilingües sí aportan).
PARAFRASIS = [
 ("hnsw",
  "HNSW builds a multi-layer proximity graph. A search enters at the top layer, walks greedily towards the target and descends layer by layer, returning approximate neighbors in logarithmic time.",
  "HNSW construye un grafo de proximidad con varias capas. La búsqueda entra por la capa superior, avanza de forma voraz hacia el objetivo y desciende capa a capa, devolviendo vecinos aproximados en tiempo logarítmico.",
  "def greedy_search(graph, vectors, q, entry):\n    best, improved = entry, True\n    while improved:\n        improved = False\n        for nb in graph[best]:\n            if dist(vectors[nb], q) < dist(vectors[best], q):\n                best, improved = nb, True\n    return best",
  "find similar vectors fast by hopping through a layered network", "buscar vectores parecidos rápido saltando por una red de niveles"),
 ("inverted_index",
  "An inverted index maps every term to the list of documents that contain it, so a keyword query only reads the posting lists of its terms instead of scanning the whole collection.",
  "Un índice invertido asocia cada término con la lista de documentos que lo contienen, de modo que una consulta solo lee las listas de sus términos en lugar de recorrer toda la colección.",
  "def build_index(docs):\n    index = {}\n    for doc_id, text in docs.items():\n        for term in set(text.lower().split()):\n            index.setdefault(term, []).append(doc_id)\n    return index",
  "data structure that avoids scanning every document for a word", "estructura que evita recorrer todos los textos al buscar una palabra"),
 ("bm25",
  "BM25 scores a document by summing, for each query term, its inverse document frequency times a saturating term frequency, normalized by document length with parameters k1 and b.",
  "BM25 puntúa un documento sumando, para cada término de la consulta, su frecuencia inversa de documento por una frecuencia de término saturada, normalizada por la longitud del documento con los parámetros k1 y b.",
  "def bm25_term(tf, idf, dl, avgdl, k1=1.2, b=0.75):\n    norm = k1 * (1 - b + b * dl / avgdl)\n    return idf * tf * (k1 + 1) / (tf + norm)",
  "classic keyword ranking formula with length normalization", "fórmula clásica de ranking por palabras clave que penaliza textos largos"),
 ("bpe",
  "Byte-pair encoding starts from single bytes and repeatedly merges the most frequent adjacent pair into a new token until the vocabulary reaches the desired size.",
  "La codificación por pares de bytes parte de bytes sueltos y fusiona repetidamente el par adyacente más frecuente en un nuevo token hasta alcanzar el tamaño de vocabulario deseado.",
  "def most_frequent_pair(ids):\n    counts = {}\n    for a, b in zip(ids, ids[1:]):\n        counts[(a, b)] = counts.get((a, b), 0) + 1\n    return max(counts, key=counts.get)",
  "how a language model splits text into subword pieces by merging common pairs", "cómo un modelo de lenguaje trocea el texto en piezas fusionando pares comunes"),
 ("cosine",
  "Cosine similarity measures the angle between two vectors: the dot product divided by the product of their norms. It ignores magnitude and ranges from -1 to 1.",
  "La similitud del coseno mide el ángulo entre dos vectores: el producto punto dividido por el producto de sus normas. Ignora la magnitud y va de -1 a 1.",
  "def cosine(a, b):\n    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))",
  "compare two vectors by direction rather than length", "comparar dos vectores por su dirección y no por su longitud"),
 ("kmeans",
  "K-means alternates two steps: assign every point to its nearest centroid, then move each centroid to the mean of its assigned points, until assignments stop changing.",
  "K-means alterna dos pasos: asignar cada punto a su centroide más cercano y mover cada centroide a la media de sus puntos, hasta que las asignaciones dejan de cambiar.",
  "def kmeans_step(X, C):\n    d = ((X[:, None] - C[None]) ** 2).sum(-1)\n    a = d.argmin(1)\n    return np.stack([X[a == k].mean(0) for k in range(len(C))]), a",
  "group unlabeled points into clusters around moving centers", "agrupar puntos sin etiquetas alrededor de centros que se desplazan"),
 ("pq",
  "Product quantization splits a vector into sub-vectors and replaces each by the index of its nearest codeword, compressing a float vector to a few bytes while allowing fast approximate distances.",
  "La cuantización de producto divide un vector en subvectores y sustituye cada uno por el índice de su palabra de código más cercana, comprimiendo un vector de floats a pocos bytes con distancias aproximadas rápidas.",
  "def pq_encode(x, codebooks):\n    subs = np.split(x, len(codebooks))\n    return [int(((cb - s) ** 2).sum(1).argmin()) for s, cb in zip(subs, codebooks)]",
  "shrink embeddings to a handful of bytes using small codebooks", "reducir embeddings a unos pocos bytes usando diccionarios pequeños"),
 ("reranker",
  "A cross-encoder reranker reads the query and a candidate document together in one transformer pass and outputs a relevance score; it is accurate but too slow for the whole corpus.",
  "Un reranker cross-encoder lee la consulta y un documento candidato juntos en una sola pasada del transformer y devuelve una puntuación de relevancia; es preciso pero demasiado lento para todo el corpus.",
  "def rerank(model, query, candidates, top_n=10):\n    scores = model.predict([(query, c) for c in candidates])\n    order = np.argsort(-scores)[:top_n]\n    return [candidates[i] for i in order]",
  "second-stage model that rescores a short list of candidates jointly with the question", "modelo de segunda etapa que reordena una lista corta mirando pregunta y texto a la vez"),
 ("matryoshka",
  "Matryoshka representation learning trains an embedding so that its first m dimensions already form a good smaller embedding, letting you truncate vectors and re-normalize them to save memory.",
  "El aprendizaje de representaciones Matryoshka entrena un embedding para que sus primeras m dimensiones ya formen un embedding más pequeño útil, permitiendo truncar los vectores y renormalizarlos para ahorrar memoria.",
  "def truncate(emb, m):\n    out = emb[:, :m]\n    return out / np.linalg.norm(out, axis=1, keepdims=True)",
  "cut a vector to fewer coordinates without retraining", "recortar un vector a menos coordenadas sin volver a entrenar"),
 ("contrastive",
  "Contrastive loss pulls the embeddings of matching pairs together and pushes apart the other examples in the batch, using a softmax over similarities scaled by a temperature.",
  "La pérdida contrastiva acerca los embeddings de los pares que coinciden y aleja el resto de ejemplos del lote, usando un softmax sobre similitudes escaladas por una temperatura.",
  "def info_nce(q, d, tau=0.05):\n    logits = (q @ d.T) / tau\n    labels = torch.arange(len(q))\n    return torch.nn.functional.cross_entropy(logits, labels)",
  "training objective that treats other items in the batch as wrong answers", "objetivo de entrenamiento que trata los demás ejemplos del lote como respuestas erróneas"),
 ("attention",
  "Self-attention lets every token build its representation as a weighted average of all value vectors, with weights given by the softmax of scaled query-key dot products.",
  "La autoatención permite que cada token construya su representación como un promedio ponderado de todos los vectores de valor, con pesos dados por el softmax de productos punto consulta-clave escalados.",
  "def attention(q, k, v):\n    w = torch.softmax(q @ k.T / q.shape[-1] ** 0.5, dim=-1)\n    return w @ v",
  "mechanism where each word looks at all the other words to update itself", "mecanismo en el que cada palabra mira a todas las demás para actualizarse"),
 ("cosine_decay",
  "A cosine learning-rate schedule starts at the peak value after warmup and decays smoothly towards zero following half a cosine wave, avoiding abrupt drops late in training.",
  "Un calendario de tasa de aprendizaje con coseno parte del valor máximo tras el calentamiento y decae suavemente hacia cero siguiendo media onda de coseno, evitando caídas bruscas al final del entrenamiento.",
  "def lr_at(step, total, peak=3e-4):\n    return 0.5 * peak * (1 + math.cos(math.pi * step / total))",
  "how to shrink the step size smoothly while a network trains", "cómo reducir poco a poco el tamaño del paso mientras una red entrena"),
 ("dropout",
  "Dropout randomly zeroes a fraction of activations during training and rescales the rest, which discourages co-adaptation of neurons and acts as a regularizer; it is disabled at inference.",
  "Dropout pone a cero al azar una fracción de las activaciones durante el entrenamiento y reescala el resto, lo que evita la coadaptación de neuronas y actúa como regularizador; se desactiva en inferencia.",
  "def dropout(x, p=0.1, training=True):\n    if not training:\n        return x\n    mask = (np.random.rand(*x.shape) > p) / (1 - p)\n    return x * mask",
  "randomly switching off neurons to reduce overfitting", "apagar neuronas al azar para reducir el sobreajuste"),
 ("layernorm",
  "Layer normalization standardizes each token's feature vector to zero mean and unit variance and then applies a learned scale and shift, stabilizing the training of deep transformers.",
  "La normalización por capas estandariza el vector de características de cada token a media cero y varianza uno y luego aplica una escala y un desplazamiento aprendidos, estabilizando el entrenamiento de transformers profundos.",
  "def layer_norm(x, g, b, eps=1e-5):\n    mu = x.mean(-1, keepdims=True)\n    var = x.var(-1, keepdims=True)\n    return g * (x - mu) / np.sqrt(var + eps) + b",
  "rescale each feature vector so deep networks train stably", "reescalar cada vector de características para que las redes profundas entrenen estables"),
 ("grad_clip",
  "Gradient clipping rescales the gradient whenever its global norm exceeds a threshold, which prevents rare huge updates from destabilizing training.",
  "El recorte de gradiente reescala el gradiente cada vez que su norma global supera un umbral, lo que evita que actualizaciones enormes y poco frecuentes desestabilicen el entrenamiento.",
  "def clip_grad(grads, max_norm=1.0):\n    total = math.sqrt(sum(float((g ** 2).sum()) for g in grads))\n    scale = min(1.0, max_norm / (total + 1e-6))\n    return [g * scale for g in grads]",
  "limit the size of parameter updates to avoid exploding values", "limitar el tamaño de las actualizaciones de parámetros para evitar valores que explotan"),
 ("lru_cache",
  "An LRU cache keeps the most recently used entries and evicts the least recently used one when it is full, giving constant-time lookups with a hash map plus a linked list.",
  "Una caché LRU conserva las entradas usadas más recientemente y expulsa la menos reciente cuando se llena, con búsquedas en tiempo constante mediante un mapa hash y una lista enlazada.",
  "from collections import OrderedDict\n\nclass LRU:\n    def __init__(self, cap):\n        self.cap, self.d = cap, OrderedDict()\n    def get(self, k):\n        self.d.move_to_end(k)\n        return self.d[k]",
  "memory that forgets whatever was touched longest ago", "memoria que olvida lo que se tocó hace más tiempo"),
 ("token_bucket",
  "A token bucket rate limiter adds tokens at a fixed rate up to a maximum capacity; each request consumes one token and is rejected when the bucket is empty, allowing short bursts.",
  "Un limitador de tasa de cubeta de tokens añade tokens a ritmo fijo hasta una capacidad máxima; cada petición consume un token y se rechaza si la cubeta está vacía, lo que permite ráfagas cortas.",
  "def allow(bucket, now, rate=5.0, cap=10.0):\n    bucket['tokens'] = min(cap, bucket['tokens'] + (now - bucket['t']) * rate)\n    bucket['t'] = now\n    if bucket['tokens'] >= 1:\n        bucket['tokens'] -= 1\n        return True\n    return False",
  "stop an API client from sending too many requests per second", "impedir que un cliente de una API envíe demasiadas peticiones por segundo"),
 ("binary_search",
  "Binary search finds a value in a sorted array by repeatedly comparing it with the middle element and discarding half of the remaining range, so it needs only logarithmic comparisons.",
  "La búsqueda binaria encuentra un valor en un arreglo ordenado comparándolo repetidamente con el elemento central y descartando la mitad del rango restante, por lo que solo necesita comparaciones logarítmicas.",
  "def binary_search(a, x):\n    lo, hi = 0, len(a) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if a[mid] == x:\n            return mid\n        lo, hi = (mid + 1, hi) if a[mid] < x else (lo, mid - 1)\n    return -1",
  "locate an item in an ordered list by halving the range each time", "localizar un elemento en una lista ordenada partiendo el rango a la mitad cada vez"),
 ("hash_collision",
  "A hash table maps keys to buckets with a hash function; when two keys land in the same bucket a collision occurs, resolved by chaining or open addressing.",
  "Una tabla hash asigna claves a cubetas mediante una función hash; cuando dos claves caen en la misma cubeta ocurre una colisión, que se resuelve con encadenamiento o direccionamiento abierto.",
  "def put(table, key, value):\n    bucket = table[hash(key) % len(table)]\n    for i, (k, _) in enumerate(bucket):\n        if k == key:\n            bucket[i] = (key, value)\n            return\n    bucket.append((key, value))",
  "what happens when two different keys map to the same slot", "qué pasa cuando dos claves distintas caen en la misma casilla"),
 ("deadlock",
  "A deadlock happens when two threads each hold a lock the other needs and wait forever; acquiring locks in a single global order prevents the circular wait.",
  "Un interbloqueo ocurre cuando dos hilos retienen cada uno un candado que el otro necesita y esperan para siempre; adquirir los candados en un único orden global evita la espera circular.",
  "def transfer(a, b, amount):\n    first, second = sorted([a, b], key=id)\n    with first.lock:\n        with second.lock:\n            a.balance -= amount\n            b.balance += amount",
  "two threads waiting on each other forever", "dos hilos esperándose mutuamente para siempre"),
 ("btree_index",
  "A B-tree index keeps keys sorted in a balanced tree with wide nodes, so a database can find a row or scan a range with only a few disk reads.",
  "Un índice B-tree mantiene las claves ordenadas en un árbol balanceado con nodos anchos, de modo que una base de datos puede encontrar una fila o recorrer un rango con pocas lecturas de disco.",
  "def create_index(cursor):\n    cursor.execute('CREATE INDEX idx_users_email ON users (email)')\n    cursor.execute('EXPLAIN SELECT * FROM users WHERE email = ?', ('a@b.c',))",
  "make database lookups on a column faster with a sorted tree", "acelerar las búsquedas de una columna en una base de datos con un árbol ordenado"),
 ("sharding",
  "Sharding splits a large dataset across several machines by a shard key, so each node stores and serves only part of the data and capacity grows horizontally.",
  "El sharding reparte un conjunto de datos grande entre varias máquinas según una clave de partición, de modo que cada nodo almacena y sirve solo una parte y la capacidad crece horizontalmente.",
  "def shard_for(key, n_shards):\n    h = int(hashlib.md5(key.encode()).hexdigest(), 16)\n    return h % n_shards",
  "distribute data over many servers so no single one holds everything", "distribuir los datos entre muchos servidores para que ninguno guarde todo"),
 ("backoff",
  "Exponential backoff retries a failed request after waiting 2^n times a base delay, usually with random jitter, so that many clients do not hammer a recovering service at once.",
  "El reintento con retroceso exponencial repite una petición fallida tras esperar 2^n veces un retardo base, normalmente con jitter aleatorio, para que muchos clientes no saturen a la vez un servicio que se recupera.",
  "def retry(fn, tries=5, base=0.5):\n    for n in range(tries):\n        try:\n            return fn()\n        except Exception:\n            time.sleep(base * 2 ** n * random.uniform(0.5, 1.5))\n    raise RuntimeError('agotados los reintentos')",
  "wait longer after each failed network call before trying again", "esperar más tras cada llamada de red fallida antes de volver a intentar"),
 ("mocking",
  "In unit tests a mock replaces a real dependency such as a network client, so the test is fast, deterministic and verifies how the code under test calls it.",
  "En las pruebas unitarias un mock sustituye una dependencia real, como un cliente de red, para que la prueba sea rápida, determinista y verifique cómo la llama el código bajo prueba.",
  "from unittest.mock import patch\n\ndef test_fetch():\n    with patch('app.http_get', return_value={'ok': True}) as m:\n        assert fetch('/x') == {'ok': True}\n        m.assert_called_once_with('/x')",
  "fake a service in a test so it runs without the internet", "simular un servicio en una prueba para que corra sin internet"),
 ("git_rebase",
  "Git rebase replays the commits of your branch on top of another base commit, producing a linear history instead of a merge commit, but it rewrites commit hashes.",
  "Git rebase reaplica los commits de tu rama encima de otro commit base, produciendo un historial lineal en lugar de un commit de fusión, pero reescribe los hashes de los commits.",
  "import subprocess\n\ndef rebase_onto_main():\n    subprocess.run(['git', 'fetch', 'origin'], check=True)\n    subprocess.run(['git', 'rebase', 'origin/main'], check=True)",
  "move my branch commits to the tip of main to keep a straight history", "mover los commits de mi rama a la punta de main para mantener un historial recto"),
 ("docker",
  "A Docker container packages an application with its dependencies and runs it as an isolated process that shares the host kernel, which makes deployments reproducible and lightweight.",
  "Un contenedor Docker empaqueta una aplicación con sus dependencias y la ejecuta como un proceso aislado que comparte el kernel del anfitrión, lo que hace los despliegues reproducibles y ligeros.",
  "import subprocess\n\ndef run_container(image, port=8000):\n    subprocess.run(['docker', 'run', '--rm', '-p', f'{port}:{port}', image], check=True)",
  "ship software with everything it needs so it runs the same everywhere", "entregar software con todo lo que necesita para que corra igual en cualquier sitio"),
 ("pagination",
  "Cursor-based pagination returns a page of results plus an opaque cursor pointing to the last item, which is more stable and efficient than offsets when the data changes between requests.",
  "La paginación por cursor devuelve una página de resultados junto con un cursor opaco que apunta al último elemento, algo más estable y eficiente que los desplazamientos cuando los datos cambian entre peticiones.",
  "def list_items(db, cursor=None, limit=20):\n    rows = db.query('SELECT * FROM items WHERE id > ? ORDER BY id LIMIT ?', cursor or 0, limit)\n    return rows, (rows[-1].id if rows else None)",
  "return long API results in chunks that stay consistent while data changes", "devolver resultados largos de una API en trozos que se mantienen coherentes mientras los datos cambian"),
 ("regex",
  "A regular expression describes a set of strings with literals, character classes and quantifiers; the engine scans the text and returns the matches, but nested quantifiers can cause catastrophic backtracking.",
  "Una expresión regular describe un conjunto de cadenas con literales, clases de caracteres y cuantificadores; el motor recorre el texto y devuelve las coincidencias, pero los cuantificadores anidados pueden causar retroceso catastrófico.",
  "import re\n\nEMAIL = re.compile(r'[\\w.+-]+@[\\w-]+\\.[\\w.]+')\n\ndef find_emails(text):\n    return EMAIL.findall(text)",
  "pattern language to extract pieces of text such as e-mail addresses", "lenguaje de patrones para extraer fragmentos de texto como direcciones de correo"),
 ("unicode_norm",
  "Unicode normalization converts equivalent character sequences to a canonical form: NFC composes accents into single code points, while NFD decomposes them into base letter plus combining mark.",
  "La normalización Unicode convierte secuencias de caracteres equivalentes a una forma canónica: NFC compone los acentos en un solo punto de código, mientras que NFD los descompone en letra base más marca combinante.",
  "import unicodedata\n\ndef strip_accents(s):\n    nfd = unicodedata.normalize('NFD', s)\n    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')",
  "make visually identical accented strings compare as equal", "lograr que cadenas con tildes visualmente idénticas se comparen como iguales"),
]
print(f"{len(PARAFRASIS)} conceptos x 3 documentos (en/es/código) en el mini golden set de paráfrasis")

In [ ]:
def corpus_parafrasis() -> dict:
    """Mini golden set de PARÁFRASIS y búsqueda CRUZADA ES<->EN<->código (29 conceptos x 3 documentos, 58 queries).
    Mismo esquema que carga_corpus(). Las queries casi no comparten palabras con sus documentos: BM25 sufre y los
    modelos semánticos/multilingües deberían ganar. Es un conjunto pequeño y escrito a mano: úsalo para diagnosticar,
    no para reportar cifras de producción."""
    docs, queries, qrels = [], [], {}
    for key, en, es, code, q_en, q_es in PARAFRASIS:
        ids = {"en": f"{key}-en", "es": f"{key}-es", "code": f"{key}-code"}
        docs += [{"id": ids["en"], "lang": "en", "text": en}, {"id": ids["es"], "lang": "es", "text": es},
                 {"id": ids["code"], "lang": "code", "text": code}]
        for lang, q in (("en", q_en), ("es", q_es)):
            qid = f"par-{key}-{lang}"
            queries.append({"qid": qid, "lang": lang, "text": q})
            qrels[qid] = {d: 1 for d in ids.values()}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "paraphrase-golden"}

In [ ]:
D1, Q1, D2, Q2 = EMB["v1"]["D"], EMB["v1"]["Q"], EMB["v2"]["D"], EMB["v2"]["Q"]
nd = lambda S: evaluate(run_desde_scores(S), qrels)["ndcg"]
pq_de = lambda S: ndcg_por_query(run_desde_scores(S))
print({"BM25": round(nd(S_BM25), 3), "v1": round(nd(Q1 @ D1.T), 3), "v2": round(nd(Q2 @ D2.T), 3)})

## 1. Árboles de decisión ejecutables
Un árbol de decisión en una diapositiva no se puede testear. Lo escribimos como **funciones** que devuelven la decisión **y sus razones**, para poder aplicarlas a escenarios, revisarlas en un PR y cambiar un umbral cuando una medición lo contradiga.

> ### 🎨 GEMINI-IMAGEN · ID: `M19-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Árbol de decisión del índice vectorial**
>
> **Qué muestra:** La versión visual del árbol `elige_indice`: de N, memoria, filtros y actualizaciones a Flat, HNSW, HNSW+int8, IVF-PQ o DiskANN.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Árbol de decisión de arriba abajo: fuerza bruta, HNSW, HNSW con int8, DiskANN o IVF-PQ según latencia, memoria y disco, con notas sobre filtros y actualizaciones.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama de árbol de decisión técnico, fondo blanco, paleta: azul #1F6FEB (preguntas), verde #2CA02C (decisiones finales), naranja #F28E2B (advertencias), gris #6B7280 (flechas). Tipografía sans-serif legible. Flujo de arriba abajo.
> Nodo raíz (rombo azul): "¿Fuerza bruta cumple el p99 con batch = 1? (mídelo)". Rama "sí" → hoja verde "Flat exacto (FAISS IndexFlatIP / numpy)". Rama "no" → rombo azul "¿Vectores float32 + grafo caben en RAM?".
> "sí" → hoja verde "HNSW (M = 16–32, efSearch según recall)". "no" → rombo azul "¿Caben en RAM con int8?".
> "sí" → hoja verde "HNSW + int8 (rescore con float en disco)". "no" → rombo azul "¿SSD rápido disponible?".
> "sí" → hoja verde "DiskANN / Vamana (PQ en RAM, grafo en SSD)". "no" → hoja verde "IVF-PQ (+ refine)".
> A la derecha, dos notas naranjas conectadas con línea discontinua a todo el árbol: "Filtros muy selectivos → pre-filtro o grafo con filtros (ACORN)" y "Muchas actualizaciones → evita reentrenar IVF; HNSW admite inserciones, compacta borrados".
> Título arriba: "¿Qué índice vectorial?".
> Evita: cifras de rendimiento, logotipos, iconos decorativos, texto pequeño.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M19-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Árbol de decisión de arriba abajo: fuerza bruta, HNSW, HNSW con int8, DiskANN o IVF-PQ según latencia, memoria y disco, con notas sobre filtros y actualizaciones.](assets/M19-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> ¿Flat cumple p99? ─sí→ Flat
>    └no→ ¿float32 cabe en RAM? ─sí→ HNSW
>           └no→ ¿int8 cabe? ─sí→ HNSW+int8 (+rescore)
>                  └no→ ¿SSD? ─sí→ DiskANN │ no→ IVF-PQ
> Notas: filtros selectivos → pre-filtro/ACORN · updates → HNSW, compactar
> ```

> 👀 **Qué debes observar:** la **primera** pregunta no es «¿qué índice es el mejor?» sino «¿necesito un índice?». El umbral de esa pregunta no se copia de un blog: lo medimos a continuación.

In [ ]:
import faiss
faiss.omp_set_num_threads(1)

def p99_ms(fn, Q, warm=5):
    for q in Q[:warm]: fn(q)
    t = []
    for q in Q:
        t0 = time.perf_counter(); fn(q); t.append((time.perf_counter() - t0) * 1e3)
    return float(np.percentile(t, 99))

rng = np.random.default_rng(SEED); d = 128
Ns = [1_000, 10_000, 50_000, 100_000] if FAST_DEV_RUN else [1_000, 10_000, 100_000, 500_000, 1_000_000]
Xall = rng.normal(size=(max(Ns), d)).astype("float32"); Xall /= np.linalg.norm(Xall, axis=1, keepdims=True)
Qs = rng.normal(size=(100, d)).astype("float32"); Qs /= np.linalg.norm(Qs, axis=1, keepdims=True)
filas = []
for N in Ns:
    X = Xall[:N]
    flat = faiss.IndexFlatIP(d); flat.add(X)
    hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); hnsw.hnsw.efConstruction = 80; hnsw.add(X); hnsw.hnsw.efSearch = 64
    gt = flat.search(Qs, 10)[1]; got = hnsw.search(Qs, 10)[1]
    rec = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(gt, got)])
    filas.append({"N": N, "flat_p99_ms": p99_ms(lambda q: flat.search(q[None], 10), Qs),
                  "hnsw_p99_ms": p99_ms(lambda q: hnsw.search(q[None], 10), Qs), "hnsw_recall@10": rec})
df_idx = pd.DataFrame(filas)
SLA_MS = 5.0
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.loglog(df_idx.N, df_idx.flat_p99_ms, "o-", color=NARANJA, label="Flat exacto (p99)"); ax.loglog(df_idx.N, df_idx.hnsw_p99_ms, "o-", color=AZUL, label="HNSW M=32 ef=64 (p99)")
ax.axhline(SLA_MS, color=ROJO, ls="--", label=f"SLA de búsqueda = {SLA_MS} ms")
ax.set_xlabel("N vectores (d=128, 1 hilo)"); ax.set_ylabel("p99 por query (ms)"); ax.set_title("¿Cuándo deja de bastar la fuerza bruta? (medido)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
df_idx.round(3)

> 👀 **Qué debes observar:** la latencia de Flat crece lineal con $N$ y cruza el SLA en algún punto entre decenas y cientos de miles de vectores **en este hardware con un hilo**; HNSW apenas crece, a cambio de un recall < 1 que **cae con N** (columna `hnsw_recall@10`): con vectores gaussianos isótropos, el peor caso para ANN, es mucho más bajo que con embeddings reales (módulo 14), y se recupera subiendo `efSearch` a costa de latencia. Nunca elijas el índice sin mirar ambas columnas. Ese cruce es el umbral `N_FLAT_MAX` que usa el árbol: **recalcúlalo** con tu $d$, tu CPU/GPU y tu SLA.

In [ ]:
N_FLAT_MAX = int(df_idx.loc[df_idx.flat_p99_ms <= SLA_MS, "N"].max()) if (df_idx.flat_p99_ms <= SLA_MS).any() else 1_000

def elige_indice(N, d, ram_gb, ssd=True, filtros_selectivos=False, muchas_actualizaciones=False, n_flat_max=N_FLAT_MAX):
    """Devuelve (decisión, razones). Memoria: float32 = 4·d B/vector, int8 = d B, grafo HNSW ≈ 2·M·4 B/vector (M=32)."""
    gb = lambda bytes_vec: N * bytes_vec / 1e9
    grafo = 2 * 32 * 4
    r = [f"N={N:,}, d={d}: float32={gb(4 * d):.1f} GB, int8={gb(d):.1f} GB, grafo≈{gb(grafo):.1f} GB, RAM={ram_gb} GB"]
    if N * d <= n_flat_max * 128:
        dec = "Flat exacto"; r.append(f"N·d ≤ umbral medido ({n_flat_max:,}×128): la fuerza bruta cumple el SLA y su recall es 1")
    elif gb(4 * d + grafo) <= 0.7 * ram_gb:
        dec = "HNSW (float32)"; r.append("cabe en RAM con margen del 30 %")
    elif gb(d + grafo) <= 0.7 * ram_gb:
        dec = "HNSW + int8 (rescore con float32 en disco)"; r.append("float32 no cabe; int8 sí (pérdida típica pequeña, módulo 12)")
    elif ssd:
        dec = "DiskANN / Vamana (PQ en RAM, grafo + vectores en SSD)"; r.append("no cabe en RAM: diseño para pocas lecturas aleatorias de SSD")
    else:
        dec = "IVF-PQ + refine"; r.append("sin SSD rápido: compresión agresiva en RAM")
    if filtros_selectivos:
        r.append("filtros selectivos: pre-filtro (si quedan pocos candidatos, fuerza bruta sobre ellos) o grafo con filtros (ACORN); nunca solo post-filtro")
    if muchas_actualizaciones:
        r.append("muchas actualizaciones: evita reentrenar centroides IVF a menudo; HNSW admite inserciones; planifica compactar borrados")
    return dec, r

for esc in [dict(N=50_000, d=384, ram_gb=16), dict(N=10_000_000, d=768, ram_gb=64), dict(N=10_000_000, d=768, ram_gb=16),
            dict(N=1_000_000_000, d=768, ram_gb=256, filtros_selectivos=True)]:
    dec, rz = elige_indice(**esc); print(f"▶ {dec}\n   " + "\n   ".join(rz))

In [ ]:
def elige_tokenizer(usa_modelo_preentrenado: bool, n_scripts: int = 1, mucho_codigo: bool = False, texto_ruidoso: bool = False):
    """Árbol de decisión del tokenizer (módulos 02–07)."""
    if usa_modelo_preentrenado:
        return "usa el tokenizer DEL MODELO", ["cambiarlo exige tokenizer transfer y reentrenar embeddings (módulo 07)",
                                               "mide fertilidad/paridad para presupuestar costo y contexto (módulo 04)"]
    r = ["byte-level o byte_fallback: ningún texto queda fuera del vocabulario (módulos 02–03)"]
    dec = "BPE byte-level"
    if n_scripts >= 3:
        dec = "BPE byte-level o Unigram con byte_fallback, vocabulario grande (≥ 100 k)"; r.append("muchos scripts: el vocabulario se reparte; mide la prima de paridad por idioma (04)")
    if mucho_codigo:
        r.append("código: pre-tokenización que preserve espacios/indentación y divida dígitos (regex tipo o200k, 02/04)")
    if texto_ruidoso:
        r.append("texto muy ruidoso o a nivel de carácter: evalúa byte-level/tokenizer-free (06–07), madurez 🔴")
    return dec, r

def elige_embedder(privacidad_on_prem: bool, multilingue: bool, pares_dominio: int, p99_ms_embed: float, presupuesto_gpu: bool):
    """Árbol de decisión del embedder (módulos 10–12). La validación final es SIEMPRE tu golden set + test pareado."""
    r = []
    if privacidad_on_prem or not presupuesto_gpu is False and p99_ms_embed < 20:
        base = "open weights autohospedado"; r.append("privacidad/latencia: el texto no sale y no hay viaje de red a una API")
    else:
        base = "API comercial o open weights"; r.append("sin restricción de residencia: compara costo total (módulo 18)")
    if multilingue:
        r.append("multilingüe: familias e5 multilingüe, BGE-M3, Qwen3-Embedding, EmbeddingGemma, Gemini Embedding (módulo 12; verifica versiones)")
    if pares_dominio >= 1000:
        r.append(f"{pares_dominio:,} pares de dominio: fine-tuning contrastivo con negativos duros filtrados (módulo 11)")
    if p99_ms_embed < 20:
        r.append("p99 de embedding < 20 ms: modelo pequeño (≤ ~300 M) o destilado; Matryoshka para bajar d")
    r.append("decide con TU golden set (estratificado por idioma) y test pareado, no con la media de MTEB (módulo 15)")
    return base, r

def elige_hibrido(identificadores_exactos: bool, parafrasis_multilingue: bool, presupuesto_ms: float, ms_reranker_por_doc: float = 2.0):
    """Árbol de decisión de la arquitectura de retrieval (módulo 13)."""
    r, etapas = [], []
    if identificadores_exactos:
        etapas.append("BM25"); r.append("identificadores, códigos de error, nombres de función: el léxico gana (módulo 13)")
    if parafrasis_multilingue or not etapas:
        etapas.append("denso"); r.append("paráfrasis y cruce ES↔EN: el denso gana")
    dec = " + ".join(etapas) + (" con RRF" if len(etapas) > 1 else "")
    K = int(max(0, presupuesto_ms - 30) // ms_reranker_por_doc)   # 30 ms reservados a primera etapa + red (supuesto)
    if K >= 20:
        dec += f" → reranker cross-encoder sobre top-{min(K, 100)}"; r.append(f"presupuesto: {presupuesto_ms} ms ⇒ K≈{K} docs a {ms_reranker_por_doc} ms/doc (supuesto)")
    return dec, r

for f, kw in [(elige_tokenizer, dict(usa_modelo_preentrenado=False, n_scripts=3, mucho_codigo=True)),
              (elige_embedder, dict(privacidad_on_prem=True, multilingue=True, pares_dominio=5000, p99_ms_embed=15, presupuesto_gpu=True)),
              (elige_hibrido, dict(identificadores_exactos=True, parafrasis_multilingue=True, presupuesto_ms=150))]:
    dec, rz = f(**kw); print(f"{f.__name__}: ▶ {dec}\n   - " + "\n   - ".join(rz))

> ⚠️ Mira con lupa `elige_embedder`: la condición `privacidad_on_prem or not presupuesto_gpu is False and p99_ms_embed < 20` tiene un **bug de precedencia** a propósito (`not presupuesto_gpu is False` se lee `not (presupuesto_gpu is False)`). La checklist de §6 existe para cazar exactamente esto: las reglas de decisión también son código y también se revisan y se testean. Ejercicio: escribe un test que lo detecte y arréglalo (el proyecto usa una versión corregida).

## 2. Trampas de evaluación, reproducidas y medidas
Cuatro trampas que hacen que un número «mejore» sin que el sistema mejore. Para cada una: el valor **reportado** con la práctica incorrecta y el valor **honesto**.

> ### 🎨 GEMINI-IMAGEN · ID: `M19-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cuatro trampas de evaluación en retrieval**
>
> **Qué muestra:** Mapa visual de las cuatro trampas que se reproducen en la sección: tuning en test, winner's curse, dependencia del golden set y sesgo de pooling, con la dirección del sesgo y su remedio.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuadrícula 2x2 con las cuatro trampas (tuning en test, winner's curse, golden set, pooling), la dirección de su sesgo y su remedio.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Infografía técnica en cuadrícula 2x2, fondo blanco, paleta: naranja #F28E2B (práctica incorrecta), azul #1F6FEB (práctica correcta), gris #6B7280 (texto secundario), verde #2CA02C (remedio). Sans-serif legible, sin decoración.
> Cuadrante 1, título "T1 · Ajustar en el test": a la izquierda una rejilla de 21 puntos con uno resaltado en naranja y la etiqueta "α elegido mirando el test"; a la derecha cinco bloques apilados (pliegues) con uno en azul y la etiqueta "validación cruzada". Flecha hacia arriba naranja "sesgo: optimista". Remedio verde: "CV o test ciego".
> Cuadrante 2, título "T2 · Winner's curse": nube de 20 puntos grises en un plano "dev" (eje x) vs "test" (eje y), uno rojo arriba a la derecha en x y en el centro en y. Flecha naranja "sesgo: optimista". Remedio: "IC pareado + Holm".
> Cuadrante 3, título "T3 · Depende del golden set": dos pequeñas tablas lado a lado, "ítem conocido (títulos)" y "paráfrasis", con dos barras cada una "BM25" y "denso" cuyo orden se invierte entre tablas. Remedio: "reportar por golden set y por slice".
> Cuadrante 4, título "T4 · Sesgo de pooling": un embudo "top-10 de BM25 → juicios", y un sistema nuevo con documentos marcados "relevante sin juzgar = 0". Flecha naranja hacia abajo "sesgo: penaliza al sistema nuevo". Remedio: "juzgar el top-k del sistema nuevo".
> Evita: cifras inventadas, logotipos, iconos decorativos, texto pequeño.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M19-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuadrícula 2x2 con las cuatro trampas (tuning en test, winner's curse, golden set, pooling), la dirección de su sesgo y su remedio.](assets/M19-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> T1 tuning en test   → optimista  → CV / test ciego
> T2 winner's curse   → optimista  → IC pareado + Holm
> T3 golden set       → cambia el orden → reportar por golden set y slice
> T4 pooling          → penaliza al nuevo → juzgar su top-k
> ```

> 👀 **Qué debes observar:** dos trampas inflan el número del sistema que más tuneaste, una cambia el veredicto según el golden set y otra castiga al sistema nuevo. Saber la **dirección** del sesgo te dice qué sistema está favorecido antes de mirar ningún número.

### T1 · Ajustar en el test
Elegimos el peso $\alpha$ de la fusión lineal $\alpha \cdot \text{denso} + (1-\alpha)\cdot\text{BM25}$ (scores min-max por query) en una rejilla de 21 valores. Práctica incorrecta: elegir $\alpha$ con **todas** las queries y reportar ese nDCG. Práctica correcta: **validación cruzada** de 5 pliegues (elegir en 4, medir en 1).

In [ ]:
def minmax(S):
    lo, hi = S.min(1, keepdims=True), S.max(1, keepdims=True)
    return (S - lo) / np.maximum(hi - lo, 1e-9)

SD, SB = minmax(Q2 @ D2.T), minmax(S_BM25)
alfas = np.linspace(0, 1, 21)
PQ_A = np.array([pq_de(a * SD + (1 - a) * SB) for a in alfas])        # (alfas, queries)
reportado = PQ_A.mean(1).max()
rng = np.random.default_rng(SEED); pliegues = np.array_split(rng.permutation(len(qids)), 5)
honesto = np.zeros(len(qids))
for f in pliegues:
    tr = np.setdiff1d(np.arange(len(qids)), f)
    a_best = PQ_A[:, tr].mean(1).argmax()
    honesto[f] = PQ_A[a_best, f]
T1 = {"reportado (α elegido en test)": reportado, "honesto (CV 5 pliegues)": honesto.mean()}
print({k: round(v, 3) for k, v in T1.items()}, "· α* en test =", alfas[PQ_A.mean(1).argmax()])

### T2 · *Winner's curse*: elegir el mejor de muchos
Generamos 20 configuraciones **casi equivalentes** (v2 restringido a 64 dimensiones aleatorias de 128: como 20 semillas o 20 combinaciones de hiperparámetros) y elegimos la mejor en un *dev* de 55 queries. ¿Cuánto rinde en el *test* (las otras 55)?

In [ ]:
rng = np.random.default_rng(SEED); perm = rng.permutation(len(qids)); dev, te = perm[:55], perm[55:]
cfg = []
for c in range(20):
    keep = rng.choice(D2.shape[1], D2.shape[1] // 2, replace=False)
    pq = pq_de(l2n(Q2[:, keep]) @ l2n(D2[:, keep]).T)
    cfg.append((pq[dev].mean(), pq[te].mean()))
cfg = np.array(cfg); b = cfg[:, 0].argmax()
T2 = {"reportado (mejor en dev)": cfg[b, 0], "honesto (ese mismo en test)": cfg[b, 1], "media de las 20 en test": cfg[:, 1].mean()}
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(cfg[:, 0], cfg[:, 1], color=GRIS, label="20 configuraciones"); ax.scatter(cfg[b, 0], cfg[b, 1], color=ROJO, s=90, label="la «ganadora» en dev")
lim = [cfg.min() - .02, cfg.max() + .02]; ax.plot(lim, lim, ls=":", color=GRIS, label="dev = test")
ax.set_xlabel("nDCG@10 en dev (55 queries)"); ax.set_ylabel("nDCG@10 en test (55 queries)"); ax.set_title("Winner's curse: la mejor en dev regresa a la media")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in T2.items()}, f"· correlación dev-test = {np.corrcoef(cfg[:, 0], cfg[:, 1])[0, 1]:.2f}")

> 👀 **Qué debes observar:** la «ganadora» en dev pierde en test buena parte de su ventaja sobre la media (**regresión a la media**): una parte de la diferencia entre configuraciones es real y otra es **ruido de muestreo de queries**. Cuanto menor la correlación dev-test, mayor el optimismo del número de dev. Con 55 queries, diferencias de ±0.02–0.05 son indistinguibles del azar (el IC del módulo 15 lo habría dicho). Remedio: test ciego que no se toca hasta el final, IC pareados y corrección por comparaciones múltiples (Holm).

### T3 · La conclusión depende del golden set
El golden set canónico del módulo 00 es de «ítem conocido» (la query es el **título** del artículo, la primera frase del docstring), que favorece a los métodos léxicos. El mini golden set de paráfrasis del módulo 13 pregunta con **otras palabras**. Mismo par de sistemas, dos golden sets:

In [ ]:
P = corpus_parafrasis(); p_ids = [d["id"] for d in P["docs"]]; p_txt = [d["text"] for d in P["docs"]]; p_q = [q["text"] for q in P["queries"]]
bmP = BM25().fit([tokenize_simple(t) for t in p_txt], p_ids)
SBP = np.array([bmP.scores(tokenize_simple(q)) for q in p_q])
SDP = VERS["v2"].encode_queries(p_q) @ VERS["v2"].encode_docs(p_txt).T
run_p = lambda S: {q["qid"]: rank_from_scores(S[i], p_ids, 10) for i, q in enumerate(P["queries"])}
T3 = {"canónico · BM25": nd(S_BM25), "canónico · denso v2": nd(Q2 @ D2.T),
      "paráfrasis · BM25": evaluate(run_p(SBP), P["qrels"])["ndcg"], "paráfrasis · denso v2": evaluate(run_p(SDP), P["qrels"])["ndcg"]}
print({k: round(v, 3) for k, v in T3.items()})
print(f"Δ(denso − BM25): canónico {T3['canónico · denso v2'] - T3['canónico · BM25']:+.3f} · paráfrasis {T3['paráfrasis · denso v2'] - T3['paráfrasis · BM25']:+.3f}")

> 👀 **Qué debes observar:** el mismo par de sistemas cambia de tamaño (y puede cambiar de signo) la diferencia según el golden set; en paráfrasis ambos caen mucho porque las queries casi no comparten palabras. ⚠️ Con el *fallback* TF-IDF+SVD el «denso» también es léxico (n-gramas de caracteres), así que **no** gana en paráfrasis; con un embedder neuronal multilingüe real, el resultado esperable (módulo 13) es que el denso gane con claridad en paráfrasis. La lección es la misma: **reporta por golden set y por slice**, nunca un único número.

### T4 · Sesgo de *pooling*: juzgar solo lo que vio el sistema viejo
En la práctica los qrels se construyen juzgando los top-$k$ de los sistemas existentes (*pooling*). Un sistema nuevo que encuentra documentos relevantes **no juzgados** recibe 0 por ellos. Simulamos que solo se juzgó el top-10 de BM25:

In [ ]:
run_bm = run_desde_scores(S_BM25)
qrels_pool = {q: {d: r for d, r in rels.items() if d in set(run_bm[q])} for q, rels in qrels.items()}
def nd_pool(S):
    run = run_desde_scores(S)
    return float(np.mean([ndcg_at_k(run[q], qrels_pool[q], 10) if any(qrels_pool[q].values()) else 0.0 for q in qids]))
T4 = {}
for nombre, S in (("BM25", S_BM25), ("v1", Q1 @ D1.T), ("v2", Q2 @ D2.T)):
    T4[nombre] = (nd(S), nd_pool(S))
print({k: (round(a, 3), round(b, 3)) for k, (a, b) in T4.items()}, "← (qrels completos, qrels del pool de BM25)")
fig, ax = plt.subplots(figsize=(10, 4))
etiquetas = ["T1 tuning en test", "T2 winner's curse", "T4 pooling (v2)"]
rep = [T1["reportado (α elegido en test)"], T2["reportado (mejor en dev)"], T4["v2"][1]]
hon = [T1["honesto (CV 5 pliegues)"], T2["honesto (ese mismo en test)"], T4["v2"][0]]
x = np.arange(3); ax.bar(x - .2, rep, .4, color=NARANJA, label="práctica incorrecta"); ax.bar(x + .2, hon, .4, color=AZUL, label="práctica correcta")
ax.set_xticks(x); ax.set_xticklabels(etiquetas); ax.set_ylabel("nDCG@10"); ax.set_ylim(min(rep + hon) - .1, max(rep + hon) + .05)
ax.set_title("Trampas de evaluación: número reportado vs número honesto"); ax.legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar:** T1 y T2 **inflan** el número (optimismo por selección); T4 **desinfla** al sistema nuevo (BM25 no cambia porque el pool es suyo, v1/v2 pierden). Las tres sesgan la decisión en direcciones previsibles: hacia el sistema que más tuneaste y hacia el sistema que construyó los juicios. Remedios: CV o test ciego (T1, T2), pools con varios sistemas + juicio de los no juzgados del sistema nuevo (T4; LLM-as-judge validado contra humanos, módulo 15).

## 3. El gap offline/online
Tres fuentes clásicas de discrepancia entre el nDCG del golden set y lo que vive el usuario:
1. **Mezcla de tráfico**: el golden set pondera los idiomas/slices distinto que el tráfico real.
2. **Interfaz**: el usuario ve 3–5 resultados (móvil, panel de RAG con contexto limitado), no 10; y hay sesgo de posición.
3. **Lo que el golden set no tiene**: queries sin respuesta en el corpus, queries ambiguas, latencia (un sistema mejor pero más lento puede perder usuarios).

🧪 (1) Recalculamos la comparación BM25 vs denso v2 ponderando cada slice por una mezcla de tráfico que varía de 0 % a 80 % de queries de código.

> ### 🎨 GEMINI-IMAGEN · ID: `M19-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Del nDCG offline a la métrica online**
>
> **Qué muestra:** Embudo que muestra por qué una mejora offline se encoge o se invierte online: mezcla de tráfico, interfaz y sesgo de posición, queries sin respuesta, latencia.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Embudo de izquierda a derecha desde el nDCG offline hasta la métrica online, con cuatro pérdidas etiquetadas y una flecha de realimentación al golden set.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Infografía técnica en forma de embudo horizontal (izquierda → derecha), fondo blanco, paleta: azul #1F6FEB (métricas), naranja #F28E2B (pérdidas), verde #2CA02C (métrica final), gris #6B7280 (texto secundario). Sans-serif legible.
> A la izquierda, una caja azul ancha: "nDCG@10 offline (golden set)". El embudo se estrecha en cuatro tramos, cada uno con una etiqueta naranja encima:
> tramo 1: "Mezcla de tráfico ≠ mezcla del golden set (reponderar por slice)";
> tramo 2: "Interfaz: el usuario ve 3–5 resultados + sesgo de posición";
> tramo 3: "Queries sin respuesta o ambiguas que el golden set no contiene";
> tramo 4: "Latencia y abandono".
> A la derecha, una caja verde estrecha: "Métrica online (éxito de sesión, interleaving, A/B)".
> Debajo, en gris, una flecha de vuelta de derecha a izquierda con el texto: "muestrear tráfico real → ampliar el golden set".
> Evita: números inventados, logotipos, iconos de personas, texto pequeño.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M19-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Embudo de izquierda a derecha desde el nDCG offline hasta la métrica online, con cuatro pérdidas etiquetadas y una flecha de realimentación al golden set.](assets/M19-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> nDCG@10 offline ▶ [mezcla de tráfico] ▶ [interfaz top-3/5 + sesgo posición] ▶ [queries sin respuesta] ▶ [latencia] ▶ métrica online
>            ◀──────────── muestrear tráfico real y ampliar el golden set ◀────────────
> ```

> 👀 **Qué debes observar:** cada tramo del embudo es una corrección que puedes **medir o simular** antes de lanzar (las celdas de abajo hacen la 1 y la 2); la flecha de vuelta es lo que cierra el gap a largo plazo.

In [ ]:
pq_bm, pq_v2 = pq_de(S_BM25), pq_de(Q2 @ D2.T)
slices = ["es", "en", "code"]
m = {n: np.array([pq[q_lang == l].mean() for l in slices]) for n, pq in (("BM25", pq_bm), ("denso v2", pq_v2))}
frac_code = np.linspace(0, 0.8, 17)
delta = []
for fc in frac_code:
    w = np.array([(1 - fc) * 0.7, (1 - fc) * 0.3, fc])          # el resto del tráfico: 70 % español, 30 % inglés (supuesto)
    delta.append(w @ m["denso v2"] - w @ m["BM25"])
w_golden = np.array([np.mean(q_lang == l) for l in slices])
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(frac_code * 100, delta, "o-", color=AZUL); ax.axhline(0, color=GRIS)
ax.axvline(w_golden[2] * 100, color=NARANJA, ls="--", label=f"mezcla del golden set ({w_golden[2]*100:.0f} % código)")
ax.set_xlabel("% de queries de código en el tráfico real"); ax.set_ylabel("Δ nDCG@10 (denso v2 − BM25) ponderado")
ax.set_title("La comparación depende de la mezcla de tráfico"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
pd.DataFrame(m, index=slices).round(3)

> 👀 **Qué debes observar:** con pesos del golden set ambos sistemas empatan; con más tráfico en español gana BM25 y con más código gana el denso. Si no sabes la mezcla real, **no sabes cuál es mejor**. En producción: pondera cada slice por su frecuencia en logs (muestreo estratificado del tráfico) y reporta también el peor slice.

🧪 (2) Interfaz: el usuario de Políglota ve 3 resultados. ¿Coincide el orden de sistemas por nDCG@10 con el orden por nDCG@3 y por la probabilidad de éxito con el modelo de clic del módulo 18?

In [ ]:
P_EXAM = np.array([0.95, 0.7, 0.5, 0.35, 0.25, 0.18, 0.13, 0.1, 0.08, 0.06])
def exito_esperado(S, top=3, p_rel=0.8, p_norel=0.05):
    """P(al menos un clic) en el top visible con el modelo de clic con sesgo de posición."""
    run = run_desde_scores(S)
    R = np.array([[1.0 if qrels[q].get(dd, 0) > 0 else 0.0 for dd in run[q][:top]] for q in qids])
    p_click = P_EXAM[:top] * np.where(R > 0, p_rel, p_norel)
    return float(np.mean(1 - np.prod(1 - p_click, axis=1)))

lo_c, esc_c = np.quantile(D2, 0.001, 0), (np.quantile(D2, 0.999, 0) - np.quantile(D2, 0.001, 0)) / 255
D2_i8 = (np.clip(np.round((D2 - lo_c) / esc_c) - 128, -128, 127) + 128) * esc_c + lo_c
sist = {"BM25": S_BM25, "v1": Q1 @ D1.T, "v2": Q2 @ D2.T, "v2 int8": Q2 @ D2_i8.T, "RRF(v2,BM25)": None, "lineal α=.5": 0.5 * SD + 0.5 * SB}
rk = lambda S: np.argsort(np.argsort(-S, 1, kind="stable"), 1, kind="stable")
sist["RRF(v2,BM25)"] = 1 / (61 + rk(Q2 @ D2.T)) + 1 / (61 + rk(S_BM25))
filas = [{"sistema": n, "nDCG@10": evaluate(run_desde_scores(S), qrels, 10)["ndcg"], "nDCG@3": evaluate(run_desde_scores(S, 3), qrels, 3)["ndcg"],
          "éxito top-3 (simulado)": exito_esperado(S)} for n, S in sist.items()]
df_gap = pd.DataFrame(filas).sort_values("nDCG@10", ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df_gap["nDCG@10"], df_gap["éxito top-3 (simulado)"], color=AZUL, s=60)
for _, r in df_gap.iterrows():
    ax.annotate(r.sistema, (r["nDCG@10"], r["éxito top-3 (simulado)"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("nDCG@10 offline"); ax.set_ylabel("P(éxito) con 3 resultados visibles"); ax.set_title("Offline vs online (modelo de clic)")
plt.tight_layout(); plt.show()
df_gap.round(3)

> 👀 **Qué debes observar:** la relación es monótona a grandes rasgos pero **no** preserva el orden fino: sistemas con nDCG@10 casi iguales pueden ordenarse distinto por nDCG@3 y por éxito en top-3, y diferencias offline de 0.01–0.02 desaparecen en la métrica online. Optimiza la métrica que se parece a la interfaz (aquí, @3) y confirma online (interleaving, módulo 18). El modelo de clic es de **juguete**: sus parámetros son supuestos.

## 4. Costos a escala: de 1 M a 1 B de documentos
Una calculadora con **supuestos explícitos** (cámbialos): embedding inicial por API (0.02 $/1M tokens, oct-2026), 400 tokens por chunk, $d = 768$, RAM a 3 $/GB-mes (**supuesto** ilustrativo para instancias con mucha memoria; mira tu proveedor), 1 réplica cada 500 QPS (**supuesto**) y 2 réplicas mínimo por disponibilidad. Comparamos float32, int8 y binario + rescore (float32 en SSD a 0.1 $/GB-mes, **supuesto**).

In [ ]:
SUP = dict(precio_tok=0.02, tok_chunk=400, d=768, usd_gb_ram_mes=3.0, usd_gb_ssd_mes=0.10, qps_por_replica=500, replicas_min=2, grafo_b=256)

def costo_sistema(N, precision, qps, sup=SUP):
    """Costo inicial (embedding) y mensual (RAM + SSD de todas las réplicas). Devuelve dict en USD."""
    bpv = {"float32": 4 * sup["d"], "int8": sup["d"], "binario+rescore": sup["d"] / 8}[precision]
    ram_gb = N * (bpv + sup["grafo_b"]) / 1e9
    ssd_gb = N * 4 * sup["d"] / 1e9 if precision != "float32" else 0.0
    rep = max(sup["replicas_min"], math.ceil(qps / sup["qps_por_replica"]))
    return {"embedding_inicial": N * sup["tok_chunk"] / 1e6 * sup["precio_tok"], "ram_mes": rep * ram_gb * sup["usd_gb_ram_mes"],
            "ssd_mes": rep * ssd_gb * sup["usd_gb_ssd_mes"], "ram_gb_por_replica": ram_gb, "replicas": rep}

Ns_c = [1e6, 1e7, 1e8, 1e9]; precs = ["float32", "int8", "binario+rescore"]
tab = pd.DataFrame([{"N": int(N), "precisión": p, **costo_sistema(N, p, qps=1000)} for N in Ns_c for p in precs])
fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(Ns_c)); w = 0.27
for j, (p, c) in enumerate(zip(precs, [GRIS, AZUL, VERDE])):
    t = tab[tab["precisión"] == p]
    ax.bar(x + (j - 1) * w, t.ram_mes + t.ssd_mes, w, color=c, label=f"{p} · infra mensual")
    ax.scatter(x + (j - 1) * w, t.embedding_inicial, color=NARANJA, marker="D", zorder=3, label="embedding inicial (una vez)" if j == 0 else None)
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(["1 M", "10 M", "100 M", "1 B"]); ax.set_xlabel("documentos (chunks)")
ax.set_ylabel("USD (log)"); ax.set_title("Costo a escala (1 000 QPS, d=768, supuestos de SUP)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
tab.round(1)

> 👀 **Qué debes observar:** el embedding inicial es un pago único que crece lineal con $N$; la **memoria se paga cada mes y por réplica**, y a partir de ~100 M de chunks domina con holgura: ahí la cuantización deja de ser una optimización y pasa a ser la arquitectura. Fíjate también en `ram_gb_por_replica`: con 1 B de vectores float32 ninguna máquina normal lo aloja → sharding (y la cola de §3 del módulo 18) o DiskANN. Todos los precios de infraestructura de esta celda son **supuestos**.

## 5. Fallos reales documentados (y su reproducción mínima)
| Fallo | Qué pasó | Fuente | Módulo |
|---|---|---|---|
| Tokens «glitch» (SolidGoldMagikarp) | tokens del vocabulario casi ausentes del entrenamiento del modelo producen comportamiento anómalo | Rumbelow & Watkins 2023 (LessWrong); Land & Bartolo 2024 (arXiv:2405.05417) | 05 |
| Aritmética sensible a la tokenización de números | la dirección de troceo de dígitos (L2R vs R2L) cambia la exactitud | Singh & Strouse 2024 (arXiv:2402.14903) | 04–05 |
| Impuesto de idioma | el mismo contenido cuesta varias veces más tokens en algunos idiomas | Petrov et al. 2023 (arXiv:2305.15425); Ahia et al. 2023 (arXiv:2305.13707) | 04 |
| Trojan Source / caracteres invisibles | Unicode bidi e invisibles hacen que lo que ves no sea lo que se procesa | Boucher & Anderson 2021 (arXiv:2111.00169) | 01, 05 |
| Retrievers densos frágiles fuera de dominio | en BEIR, BM25 supera a muchos densos zero-shot | Thakur et al. 2021 (arXiv:2104.08663) | 00, 15 |
| Límite de dimensión | hay conjuntos top-k que un embedding de dimensión fija no puede representar | Weller et al. 2025 (arXiv:2508.21038) | 15 |
| Inversión de embeddings | los vectores permiten reconstruir el texto | Morris et al. 2023 (arXiv:2310.06816) | 17–18 |
| Contexto largo en RAG | los LLMs usan peor la información en mitad del contexto | Liu et al. 2023 (arXiv:2307.03172) | 13, 20 |

Los fallos de **ingeniería** más frecuentes no salen en papers porque son silenciosos. Reproducimos cuatro con el corpus de Políglota (todos dan resultados «razonables», ninguno lanza una excepción):

In [ ]:
rng = np.random.default_rng(SEED)
sano = {"denso v2": nd(Q2 @ D2.T), "BM25": nd(S_BM25)}
rotos = {}
# F1 · Truncado silencioso al indexar (max_seq_length corto; aquí 200 caracteres)
rotos["F1 truncado al indexar"] = ("denso v2", nd(Q2 @ VERS["v2"].encode_docs([t[:200] for t in doc_texts]).T))
# F2 · Producto punto con vectores NO normalizados cuya norma crece con la longitud del documento
L = np.array([len(t) for t in doc_texts], float)
rotos["F2 dot sin normalizar"] = ("denso v2", nd(Q2 @ (D2 * np.sqrt(L / L.mean())[:, None]).T))
# F3 · Tokenización distinta al indexar (quita tildes) y al consultar (las conserva)
bm_fold = BM25().fit([tokenize_simple(t, accents="fold") for t in doc_texts], doc_ids)
rotos["F3 tildes: índice≠query"] = ("BM25", nd(np.array([bm_fold.scores(tokenize_simple(q)) for q in q_texts])))
# F4 · Caché sin versión tras reentrenar (rotación ortogonal = mismo modelo, otro espacio; módulo 18)
Rm, _ = np.linalg.qr(rng.normal(size=(D2.shape[1],) * 2))
rotos["F4 caché sin versión"] = ("denso v2", nd(Q2 @ (D2 @ Rm).T))
fig, ax = plt.subplots(figsize=(9, 4))
nombres = list(rotos); x = np.arange(len(nombres))
ax.bar(x - .2, [sano[rotos[n][0]] for n in nombres], .4, color=AZUL, label="sano")
ax.bar(x + .2, [rotos[n][1] for n in nombres], .4, color=ROJO, label="con el fallo")
ax.set_xticks(x); ax.set_xticklabels(nombres, fontsize=8); ax.set_ylabel("nDCG@10"); ax.set_title("Galería de fallos silenciosos (corpus Políglota)")
ax.legend(); plt.tight_layout(); plt.show()
print({n: f"{sano[s]:.3f} → {v:.3f}" for n, (s, v) in rotos.items()})

> 👀 **Qué debes observar:** F1–F3 cuestan unos pocos puntos de nDCG: lo bastante para perder un A/B, demasiado poco para que alguien abra un incidente. F4 hunde la calidad a cero, pero solo para las queries servidas desde caché (en producción: las más populares). Ninguno lanza excepciones; todos se detectan con **tests de contrato** (misma normalización/tokenización en ambos lados, norma de los vectores ≈ 1, versión en la clave) y con un golden set ejecutado en CI.

## 6. Checklist de revisión de PRs (ejecutable)
La checklist humana (para la descripción del PR):
1. **Contrato de versión**: ¿cambia modelo, checkpoint, servidor, normalización, truncado, prefijos o chunking? → es una **migración** (módulo 18), no un PR normal.
2. **Simetría**: misma normalización Unicode y tokenización en indexación y consulta; prefijos `query:`/`passage:` correctos.
3. **Vectores**: normalizados si la métrica es coseno/producto; dimensión y dtype comprobados al cargar.
4. **Evaluación**: golden set con test ciego, IC pareado vs producción, slices por idioma; hiperparámetros elegidos fuera del test.
5. **Costo y latencia**: tokens/s medidos, p99 con batch = 1, memoria proyectada a N de producción.
6. **Privacidad**: ¿se envía texto a terceros? ¿los vectores nuevos heredan la clasificación? ¿el borrado se propaga?
7. **Rollback**: alias o flag para volver atrás en minutos.

Y su versión ejecutable: un *linter* sobre la configuración que acompaña al PR.

In [ ]:
REGLAS = [
    ("ALTA", "versión fuera de la clave de caché", lambda c: not c.get("version_en_clave_cache", False)),
    ("ALTA", "normalización Unicode distinta en índice y query", lambda c: c.get("norm_indice") != c.get("norm_query")),
    ("ALTA", "hiperparámetros elegidos en el test", lambda c: c.get("tuning_split") == "test"),
    ("ALTA", "cambio de modelo sin plan de migración (alias/rollback)", lambda c: c.get("cambia_modelo") and not c.get("plan_rollback")),
    ("MEDIA", "vectores sin normalizar con métrica coseno/IP", lambda c: c.get("metrica") in ("cos", "ip") and not c.get("normaliza", False)),
    ("MEDIA", "max_seq_length < percentil 95 de longitud de los chunks", lambda c: c.get("max_seq_len", 1e9) < c.get("p95_tokens_chunk", 0)),
    ("MEDIA", "sin IC pareado contra producción", lambda c: not c.get("ic_pareado", False)),
    ("MEDIA", "sin desglose por idioma/slice", lambda c: not c.get("slices", False)),
    ("BAJA", "prefijos query/passage ausentes en un modelo que los requiere", lambda c: c.get("modelo_requiere_prefijos") and not c.get("usa_prefijos")),
    ("BAJA", "texto enviado a API externa sin revisión de residencia", lambda c: c.get("api_externa") and not c.get("revision_privacidad")),
]

def revisa_pr(config: dict) -> pd.DataFrame:
    hall = [{"severidad": s, "hallazgo": m} for s, m, regla in REGLAS if regla(config)]
    return pd.DataFrame(hall) if hall else pd.DataFrame([{"severidad": "—", "hallazgo": "sin hallazgos ✅"}])

PR_123 = dict(cambia_modelo=True, plan_rollback=False, version_en_clave_cache=False, norm_indice="NFC", norm_query="ninguna",
              tuning_split="test", metrica="ip", normaliza=True, max_seq_len=128, p95_tokens_chunk=310, ic_pareado=False, slices=True,
              modelo_requiere_prefijos=True, usa_prefijos=True, api_externa=False)
PR_124 = dict(cambia_modelo=True, plan_rollback=True, version_en_clave_cache=True, norm_indice="NFC", norm_query="NFC", tuning_split="dev",
              metrica="cos", normaliza=True, max_seq_len=512, p95_tokens_chunk=310, ic_pareado=True, slices=True,
              modelo_requiere_prefijos=True, usa_prefijos=True, api_externa=True, revision_privacidad=True)
print("PR #123"); display(revisa_pr(PR_123)); print("PR #124"); display(revisa_pr(PR_124))

## 7. Cultura de laboratorios y equipos top (lo que se repite)
Sin atribuir prácticas internas que no están publicadas, lo que sí se lee en sus papers, informes técnicos y blogs de ingeniería:
* **La evaluación se escribe antes que el modelo**: golden set, slices y criterio de aceptación fijados de antemano (los informes de MTEB/MMTEB, BEIR y los *technical reports* de embedders publican por tarea e idioma, no un único número).
* **Baselines fuertes y honestos**: BM25 bien configurado, el modelo de producción actual, y la versión «sin el truco». Una mejora sin baseline fuerte no es una mejora.
* **Ablaciones de una variable** y semillas múltiples con IC: si el efecto es menor que la varianza entre semillas, no hay efecto.
* **Mirar los datos**: leer 50 errores antes de cambiar la arquitectura (el análisis por slices del módulo 15 es la versión sistemática).
* **Documentos de diseño y post-mortems sin culpables**: las decisiones (§1) se escriben con sus alternativas y se revisan cuando una medición las contradice.
* **Etiquetar la madurez**: separar lo sólido de lo prometedor (🟢/🟡/🔴) es parte del rigor, no una opinión.

## 🏭 En producción
* **Fusión híbrida** como opción de primera clase: RRF disponible en Elasticsearch/OpenSearch y en vector DBs como Qdrant y Weaviate (documentación oficial); Bruch et al. (2022) analizan cuándo la combinación convexa supera a RRF.
* **Reranking en dos etapas**: el patrón retrieve → rerank (Nogueira & Cho 2019) sigue siendo la forma estándar de gastar cómputo donde importa; el presupuesto de latencia fija K (§1, `elige_hibrido`).
* **Evaluación con humanos + LLM**: los juicios con LLM aceleran el pooling, pero su validez depende de calibrarlos con humanos (Thomas et al. 2023 frente a Soboroff 2024).
* **Interleaving online** para cribar y A/B para confirmar (Chapelle et al. 2012; Kohavi et al. 2020).
* **Costos**: la memoria por réplica domina a gran escala; la cuantización (int8/binaria + rescoring) y Matryoshka son las palancas principales (módulos 12 y 18).

## 🧠 Secretos de la élite
1. **Un umbral sin medición es una opinión**: `N_FLAT_MAX` sale de §1 en *tu* hardware; los árboles de decisión se versionan y se testean como código.
2. **El optimismo por selección es proporcional a cuántas cosas probaste**: reporta cuántas configuraciones evaluaste y usa test ciego (T1, T2).
3. **Los qrels tienen dueño**: si los construyó el sistema viejo, el nuevo parte con desventaja (T4); juzga los documentos nuevos del top-k antes de concluir.
4. **Pondera por tráfico, reporta el peor slice**: la media del golden set no es tu producto (§3).
5. **Los fallos caros son silenciosos**: tests de contrato (normalización, tokenización, norma, versión) en CI, no solo métricas (§5).
6. **A gran escala la arquitectura la decide la memoria por réplica**, no la calidad del embedder (§4).
7. **Sospecha de mejoras < IC**: si la diferencia pareada no excluye el 0, la respuesta correcta a «¿cuál es mejor?» es «no lo sabemos todavía».

## ⚠️ Errores comunes
* Elegir arquitectura por popularidad («todos usan HNSW») en vez de por N, memoria, filtros y actualizaciones.
* Comparar sistemas con golden sets distintos, o con un golden set construido a partir de uno de ellos.
* Reportar el mejor de N experimentos sin decir N.
* Optimizar nDCG@10 cuando la interfaz muestra 3 resultados.
* Calcular el costo de embedding y olvidar la memoria mensual por réplica.
* Revisar PRs de embeddings como si fueran PRs de código normal (sin contrato de versión ni evaluación pareada).

## 📝 Autoevaluación
<details><summary><b>1 · (Cálculo)</b> 200 M de chunks, d = 1024. ¿RAM por réplica para float32, int8 y binario (sin grafo)? ¿Cuál eliges con 128 GB por máquina?</summary>

float32: $2\cdot10^8 \cdot 4096$ B = **819 GB**; int8: **205 GB**; binario: **25.6 GB**. Con 128 GB solo cabe binario (+ grafo) en una réplica: binario en RAM + rescore con int8/float en SSD, o DiskANN; Matryoshka a 512 d reduce todo a la mitad.
</details>
<details><summary><b>2 · (Diagnóstico)</b> Tu nuevo embedder gana +0.04 nDCG@10 en dev tras probar 40 configuraciones, y +0.005 en el test ciego. ¿Qué pasó?</summary>

*Winner's curse* (T2): la ganancia en dev era sobre todo ruido de selección entre 40 configuraciones. El número válido es el del test ciego, con su IC pareado; probablemente no excluye el 0.
</details>
<details><summary><b>3 · (Diagnóstico)</b> Los juicios de relevancia vienen del top-10 de BM25 de hace un año. El nuevo denso «pierde» 0.03. ¿Qué haces antes de descartarlo?</summary>

Sospechar sesgo de pooling (T4): juzgar (humanos o LLM validado) los documentos del top-10 del denso que no tienen juicio, y recalcular. Comparar también solo sobre queries cuyo pool incluye ambos sistemas.
</details>
<details><summary><b>4 · (Transferencia)</b> Un cliente tiene 30 k documentos, 5 QPS y exige on-prem. Aplica los árboles.</summary>

Índice: Flat exacto (N muy por debajo de `N_FLAT_MAX`, recall 1, sin mantenimiento). Embedder: open weights autohospedado, multilingüe y pequeño; fine-tuning solo si hay ≥ 1 k pares. Híbrido: BM25 + denso con RRF; reranker si el presupuesto lo permite. Lo caro aquí no es la infraestructura sino el golden set.
</details>
<details><summary><b>5 · (Transferencia)</b> ¿Qué regla añadirías al linter de PRs para el fallo F1 (truncado)? ¿Y para F2?</summary>

F1: `max_seq_len ≥ p95 de longitud de chunk en tokens del tokenizer del modelo` (ya está como MEDIA). F2: test de contrato que compruebe $\lVert v\rVert \approx 1$ en una muestra del índice si la métrica es IP/coseno, y que la norma no correlacione con la longitud.
</details>
<details><summary><b>6 · (Cálculo)</b> En T1, ¿por qué el número «honesto» de la CV no es el rendimiento del α final?</summary>

La CV estima el rendimiento esperado del **procedimiento** «elegir α en datos de entrenamiento»; el α final (elegido con todo) puede ser distinto en cada pliegue. Para reportar el α final hace falta un test que no se usó para elegirlo.
</details>

## 📚 Referencias
* Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models*. https://arxiv.org/abs/2104.08663
* Cormack, G., Clarke, C., Büttcher, S. (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods*. SIGIR.
* Bruch, S., Gai, S., Ingber, A. (2022). *An Analysis of Fusion Functions for Hybrid Retrieval*. https://arxiv.org/abs/2210.11934
* Nogueira, R. & Cho, K. (2019). *Passage Re-ranking with BERT*. https://arxiv.org/abs/1901.04085
* Weller, O., Boratko, M., Naim, I., Lee, J. (2025). *On the Theoretical Limitations of Embedding-Based Retrieval*. https://arxiv.org/abs/2508.21038
* Thomas, P. et al. (2023). *Large language models can accurately predict searcher preferences*. https://arxiv.org/abs/2309.10621 · Soboroff, I. (2024). *Don't Use LLMs to Make Relevance Judgments*. https://arxiv.org/abs/2409.15133
* Smucker, M., Allan, J., Carterette, B. (2007). *A Comparison of Statistical Significance Tests for Information Retrieval Evaluation*. CIKM.
* Land, S. & Bartolo, M. (2024). *Fishing for Magikarp: Automatically Detecting Under-trained Tokens in Large Language Models*. https://arxiv.org/abs/2405.05417
* Singh, A. & Strouse, D. J. (2024). *Tokenization counts: the impact of tokenization on arithmetic in frontier LLMs*. https://arxiv.org/abs/2402.14903
* Petrov, A. et al. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages*. https://arxiv.org/abs/2305.15425
* Boucher, N. & Anderson, R. (2021). *Trojan Source: Invisible Vulnerabilities*. https://arxiv.org/abs/2111.00169
* Morris, J. X. et al. (2023). *Text Embeddings Reveal (Almost) As Much As Text*. https://arxiv.org/abs/2310.06816
* Liu, N. F. et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts*. https://arxiv.org/abs/2307.03172
* Chapelle, O. et al. (2012). *Large-scale validation and analysis of interleaved search evaluation*. ACM TOIS. · Kohavi, R., Tang, D., Xu, Y. (2020). *Trustworthy Online Controlled Experiments*. Cambridge UP.
* Patel, L. et al. (2024). *ACORN: Performant and Predicate-Agnostic Search Over Vector Embeddings and Structured Data*. https://arxiv.org/abs/2403.04871
* Material del curso: `extras/GUIA_ENTREVISTAS.md` (preguntas con respuestas modelo y rúbrica) y `extras/LECTURA_PAPERS.md` (lectura guiada por módulo).